# NMT English → Vietnamese — IWSLT'15Chạy trọn gói cả hai case rồi xuất bảng so sánh.| Bước | Thời gian trên T4 ||---|---|| 1–5 chuẩn bị | ~3 phút || 6 kiểm tra | ~5 phút || 7 train Transformer | ~75 phút (30 epoch × 2.4 phút) || 8 train Seq2Seq | ~45 phút (early stop quanh epoch 17) || 9–11 đánh giá + so sánh | ~15 phút |Tổng khoảng 2.5 giờ, vừa một phiên Colab free. Bị ngắt giữa chừng thì chạy ô 8bhoặc 9b để train tiếp.Trước khi bắt đầu: **Runtime → Change runtime type → T4 GPU**.

## 1. Kiểm tra GPU

In [ ]:
!nvidia-smiimport torchassert torch.cuda.is_available(), "Chưa bật GPU! Runtime -> Change runtime type -> T4 GPU"print(f"\nGPU     : {torch.cuda.get_device_name(0)}")print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB")print(f"PyTorch : {torch.__version__}")

## 2. Lấy code từ GitHubSửa `REPO` rồi chạy. Ô này dùng được cả lần đầu (clone) lẫn những lần sau (pull),nên mỗi khi sửa code ở máy và push lên thì chạy lại ô này là xong.`reset --hard` chỉ ghi đè file đã commit; `runs/` nằm trong `.gitignore` nêncheckpoint và kết quả train vẫn còn.

In [ ]:
# ============ SỬA DÒNG NÀY ============REPO   = "https://github.com/<user>/<repo>.git"BRANCH = "main"# ======================================import pathlibNAME = REPO.rstrip("/").split("/")[-1].replace(".git", "")ROOT = pathlib.Path("/content") / NAMEif (ROOT / ".git").exists():    print(f"Đã có {ROOT}, lấy code mới nhất")    !git -C {ROOT} fetch -q origin {BRANCH}    !git -C {ROOT} reset -q --hard origin/{BRANCH}else:    !git clone -q -b {BRANCH} {REPO} {ROOT}%cd {ROOT}!git log -1 --format="commit %h  |  %s  |  %cr"print()!ls# Repo private:  REPO = "https://<TOKEN>@github.com/<user>/<repo>.git"

## 3. Cài thư viện\n\nColab đã có PyTorch, chỉ cần thêm hai package nhẹ.

In [ ]:
!pip install -q sentencepiece 'sacrebleu>=2.4' && echo xong

## 4. Dữ liệu và tokenizerRepo đã commit sẵn corpus (31 MB) và tokenizer, nên hai ô này chủ yếu để verify.Script tự nhận ra file đã có và bỏ qua phần tải.

In [ ]:
!bash scripts/download_data.sh

In [ ]:
import osif os.path.exists('data/tokenizer/spm_vi.model'):    print("Tokenizer đã có sẵn trong repo, bỏ qua.")    print("Muốn train lại (ví dụ đổi vocab size) thì chạy:")    print("    !python scripts/prepare.py --vocab-size 16000")else:    !python scripts/prepare.py

## 5. Kiểm tra trước khi trainHai bước này rẻ và bắt lỗi trước khi đốt hàng giờ GPU.- `check_amp.py` chạy cả hai model dưới fp16. Lỗi lệch kiểu fp16/fp32 không hiện  ra khi test trên CPU nhưng làm hỏng ngay lần train đầu trên GPU.- `sanity_check.py` bắt model học thuộc 120 câu; cài đặt đúng phải đạt BLEU > 80.Ô nào báo lỗi thì dừng, đừng chạy tiếp.

In [ ]:
!python scripts/check_amp.py

In [ ]:
!python scripts/sanity_check.py

## 6. Train Case 1 — Transformer6+6 lớp, d_model=512, 4 head, d_ff=1024, dropout 0.3, pre-norm, fp16, batch gomtheo token. Tối đa 30 epoch, early stop khi BLEU dev đứng yên 5 epoch.Khoảng 2–4 phút/epoch. BLEU dev chấm sau mỗi epoch nên theo dõi được ngay. Hết ônày là đã có BLEU trên tst2013 cho greedy và beam=5.

In [ ]:
!python translate_transformers/train.py

Bị ngắt giữa chừng thì chạy ô dưới để train tiếp từ `runs/transformer/last.pt`.

In [ ]:
# !python translate_transformers/train.py --resume true

## 7. Train Case 2 — Seq2Seq + AttentionEncoder 1 lớp bi-LSTM 512, decoder 2 lớp LSTM 512, Luong attention `general`,input feeding, dropout 0.2.Khoảng 2.5 phút/epoch trên T4, tức gần bằng Case 1 dù decoder chạy tuần tự.Model này chỉ có 20.0M tham số so với 39.7M nên nửa khối lượng tính toán mỗitoken, vừa đủ bù phần thiệt do input feeding. Thường hội tụ quanh epoch 12–15 vàearly stop tự kích hoạt, nên tổng thời gian khoảng 45 phút.

In [ ]:
!python translate_seq2seq/train.py# Muốn nhanh hơn:            !python translate_seq2seq/train.py --epochs 15# Tái hiện đúng tensorflow/nmt: !python translate_seq2seq/train.py --optimizer sgd --lr 1.0 --epochs 12

Bị ngắt thì chạy ô dưới.

In [ ]:
# !python translate_seq2seq/train.py --resume true

## 8. Đánh giá thêm`train.py` đã chấm greedy và beam mặc định trên tst2013 rồi. Ô này để thử beamkhác hoặc xem câu dịch mẫu mà không phải train lại.

In [ ]:
!python evaluate.py --model transformer --beam 5  --show 5print("=" * 78)!python evaluate.py --model seq2seq     --beam 10 --show 5

## 9. Bảng so sánh + biểu đồ

In [ ]:
!python compare.py --plots

In [ ]:
from IPython.display import Image, Markdown, displayimport osif os.path.exists('runs/comparison.png'):    display(Image('runs/comparison.png'))if os.path.exists('runs/COMPARISON.md'):    display(Markdown(open('runs/COMPARISON.md').read()))

## 10. Bảng số liệu gọn để dán vào báo cáo

In [ ]:
import json, pathlibrows = []for name in ("transformer", "seq2seq"):    p = pathlib.Path("runs") / name / "benchmark.json"    if not p.exists():        print(f"chưa có {p}"); continue    r = json.loads(p.read_text())    s, t = r["summary"], r["summary"].get("test", {})    beam = next((v for k, v in t.items() if k.startswith("beam")), {})    rows.append({        "Model": name,        "BLEU greedy": t.get("greedy", {}).get("bleu_tokenized"),        "BLEU beam": beam.get("bleu_tokenized"),        "BLEU detok": beam.get("bleu_detok"),        "chrF2": beam.get("chrf2"),        "Tham số (M)": round(s.get("params_total", 0) / 1e6, 1),        "Giây/epoch": s.get("median_epoch_time_sec"),        "Tổng train (phút)": round(s.get("train_wallclock_sec", 0) / 60, 1),        "Đỉnh VRAM (GB)": s.get("peak_gpu_mem_gb"),        "ms/câu": s.get("decode_ms_per_sentence"),        "Epoch tốt nhất": s.get("best_epoch"),    })try:    import pandas as pd    df = pd.DataFrame(rows).set_index("Model").T    display(df)    print("\n--- Markdown để dán vào báo cáo ---\n")    print(df.to_markdown())except ImportError:    print(json.dumps(rows, indent=2, ensure_ascii=False))

## 11. Tải kết quả về máyTải trước khi phiên Colab hết hạn. File zip gồm `benchmark.json`, `history.csv`,câu dịch trên tst2013, biểu đồ và `best.pt`. Bỏ `last.pt` vì nó nặng gấp ba lầnmà chỉ dùng để train tiếp.

In [ ]:
!zip -qr runs.zip runs -x '*/last.pt'!du -h runs.zipfrom google.colab import filesfiles.download('runs.zip')